<a href="https://colab.research.google.com/github/Azylover/UTH_Zarzadzanie_projektami/blob/main/L01_Prokopiuk_34339_AI1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Proste rozwiązanie dla scentralizowanej dokumentacji pacjentów

Problem:

Obecnie każda placówka Medica+ prowadzi własną bazę danych pacjentów. Kiedy pacjent odwiedza inną klinikę, lekarz nie ma dostępu do jego wcześniejszej historii medycznej, co prowadzi do konieczności powtarzania pytań i przynoszenia papierowych wyników badań.

Rozwiązanie:

Poniższy kod przedstawia bardzo uproszczony model, który symuluje scentralizowaną bazę danych pacjentów i ich dokumentacji medycznej. Pozwala on na:

1.  **Reprezentowanie pacjentów:** Klasa `Patient` przechowuje podstawowe dane pacjenta.
2.  **Reprezentowanie wizyt/rekordów medycznych:** Klasa `MedicalRecord` przechowuje szczegóły dotyczące konkretnej wizyty lekarskiej (data, klinika, opis).
3.  **Zarządzanie bazą danych:** Klasa `ClinicDatabase` jest centralnym repozytorium, które przechowuje wszystkich pacjentów i ich rekordy, umożliwiając łatwy dostęp do pełnej historii medycznej niezależnie od kliniki, w której pacjent był ostatnio leczony.

Ten kod to punkt wyjścia do stworzenia bardziej złożonego systemu, który mógłby w przyszłości obejmować interfejs użytkownika, integrację z bazami danych (np. SQL) oraz zaawansowane funkcje bezpieczeństwa i autoryzacji.

In [ ]:
import datetime

class Patient:
    def __init__(self, patient_id, first_name, last_name, date_of_birth):
        self.patient_id = patient_id
        self.first_name = first_name
        self.last_name = last_name
        self.date_of_birth = date_of_birth
        self.medical_records = []

    def add_medical_record(self, record):
        if isinstance(record, MedicalRecord):
            self.medical_records.append(record)
        else:
            raise TypeError("Record must be an instance of MedicalRecord")

    def get_full_name(self):
        return f"{self.first_name} {self.last_name}"

    def __str__(self):
        return f"Patient ID: {self.patient_id}, Name: {self.get_full_name()}, DoB: {self.date_of_birth}"

class MedicalRecord:
    def __init__(self, record_id, clinic_name, doctor_name, visit_date, diagnosis, prescription=None):
        self.record_id = record_id
        self.clinic_name = clinic_name
        self.doctor_name = doctor_name
        self.visit_date = visit_date
        self.diagnosis = diagnosis
        self.prescription = prescription

    def __str__(self):
        return (f"  Record ID: {self.record_id}, Clinic: {self.clinic_name}, Doctor: {self.doctor_name}, "
                f"Date: {self.visit_date}, Diagnosis: {self.diagnosis}, Prescription: {self.prescription or 'N/A'}")

class ClinicDatabase:
    def __init__(self):
        self.patients = {}
        self.next_patient_id = 1
        self.next_record_id = 1

    def add_patient(self, first_name, last_name, date_of_birth):
        patient_id = f"P{self.next_patient_id:04d}"
        new_patient = Patient(patient_id, first_name, last_name, date_of_birth)
        self.patients[patient_id] = new_patient
        self.next_patient_id += 1
        return new_patient

    def get_patient(self, patient_id):
        return self.patients.get(patient_id)

    def add_patient_medical_record(self, patient_id, clinic_name, doctor_name, visit_date, diagnosis, prescription=None):
        patient = self.get_patient(patient_id)
        if patient:
            record_id = f"R{self.next_record_id:05d}"
            new_record = MedicalRecord(record_id, clinic_name, doctor_name, visit_date, diagnosis, prescription)
            patient.add_medical_record(new_record)
            self.next_record_id += 1
            return new_record
        else:
            print(f"Error: Patient with ID {patient_id} not found.")
            return None

    def display_patient_history(self, patient_id):
        patient = self.get_patient(patient_id)
        if patient:
            print(f"\n--- Full Medical History for {patient.get_full_name()} (ID: {patient.patient_id}) ---")
            if patient.medical_records:
                # Sort records by date for better readability
                sorted_records = sorted(patient.medical_records, key=lambda x: x.visit_date)
                for record in sorted_records:
                    print(record)
            else:
                print("No medical records found for this patient.")
            print("---------------------------------------------------")
        else:
            print(f"Patient with ID {patient_id} not found.")


### Przykładowe użycie systemu

In [ ]:
# Inicjalizacja bazy danych
medica_db = ClinicDatabase()

# Dodawanie pacjentów
patient1 = medica_db.add_patient("Anna", "Kowalska", datetime.date(1985, 3, 15))
patient2 = medica_db.add_patient("Piotr", "Nowak", datetime.date(1972, 11, 22))

print("Dodani pacjenci:")
print(patient1)
print(patient2)

# Dodawanie rekordów medycznych dla Anny Kowalskiej w różnych klinikach
medica_db.add_patient_medical_record(
    patient1.patient_id,
    "Medica+ Centrum",
    "Dr. Zieliński",
    datetime.date(2023, 1, 10),
    "Przeziębienie",
    "Paracetamol 500mg, 2x dziennie"
)

medica_db.add_patient_medical_record(
    patient1.patient_id,
    "Medica+ Wola",
    "Dr. Kowalczyk",
    datetime.date(2023, 2, 5),
    "Kontrola po przeziębieniu",
    "Brak"
)

medica_db.add_patient_medical_record(
    patient1.patient_id,
    "Medica+ Centrum",
    "Dr. Kamiński",
    datetime.date(2023, 5, 20),
    "Ból głowy",
    "Ibuprofen"
)

# Dodawanie rekordu dla Piotra Nowaka
medica_db.add_patient_medical_record(
    patient2.patient_id,
    "Medica+ Mokotów",
    "Dr. Lewandowska",
    datetime.date(2023, 4, 1),
    "Wysokie ciśnienie",
    "Lek na nadciśnienie"
)

# Wyświetlanie pełnej historii medycznej pacjenta (dostępnej z dowolnego miejsca)
medica_db.display_patient_history(patient1.patient_id)
medica_db.display_patient_history(patient2.patient_id)

# Próba pobrania nieistniejącego pacjenta
medica_db.display_patient_history("P9999")

Dodani pacjenci:
Patient ID: P0001, Name: Anna Kowalska, DoB: 1985-03-15
Patient ID: P0002, Name: Piotr Nowak, DoB: 1972-11-22

--- Full Medical History for Anna Kowalska (ID: P0001) ---
  Record ID: R00001, Clinic: Medica+ Centrum, Doctor: Dr. Zieliński, Date: 2023-01-10, Diagnosis: Przeziębienie, Prescription: Paracetamol 500mg, 2x dziennie
  Record ID: R00002, Clinic: Medica+ Wola, Doctor: Dr. Kowalczyk, Date: 2023-02-05, Diagnosis: Kontrola po przeziębieniu, Prescription: Brak
  Record ID: R00003, Clinic: Medica+ Centrum, Doctor: Dr. Kamiński, Date: 2023-05-20, Diagnosis: Ból głowy, Prescription: Ibuprofen
---------------------------------------------------

--- Full Medical History for Piotr Nowak (ID: P0002) ---
  Record ID: R00004, Clinic: Medica+ Mokotów, Doctor: Dr. Lewandowska, Date: 2023-04-01, Diagnosis: Wysokie ciśnienie, Prescription: Lek na nadciśnienie
---------------------------------------------------
Patient with ID P9999 not found.
